# 토크나이저 학습 (KoWiki)

**한국어 위키피디아(KoWiki)**로 BPE 토크나이저를 학습한다.
사전학습/파인튜닝과 분리된 **1회성** 노트북 — 여기서 만든 `tokenizer.json` 을
`train_model.ipynb` 가 로드해서 그대로 재사용한다 (토크나이저는 한 번 학습 후 고정).

## 0. 환경 설정

In [ ]:
!pip install datasets tqdm -q

In [ ]:
import os, sys, shutil, importlib

REPO_URL  = "https://github.com/kkkk2058/korean-chatbot"
REPO_DIR  = "korean-chatbot"
STAGE_DIR = f"{REPO_DIR}/stage1_from_scratch"
DRIVE_DIR = "/content/drive/MyDrive/korean_chatbot"

WIKI_DOCS = 30_000   # 토크나이저 학습에 쓸 위키 문서 수

from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL
else:
    !git -C $REPO_DIR pull

sys.path.insert(0, STAGE_DIR)
print("레포 + Drive 준비 완료")

## 1. 위키 코퍼스 수집

In [ ]:
import src.tokenizer
importlib.reload(src.tokenizer)
from src.tokenizer import BPETokenizer
import config
from tqdm.notebook import tqdm


def load_wiki_texts(n_docs):
    """위키피디아 n_docs개 문서를 스트리밍으로 받아 줄 단위 텍스트 리스트로 반환.
    streaming=True 라 전체(수 GB) 다운로드 없이 앞부분만 받음."""
    from datasets import load_dataset
    wiki = load_dataset("wikimedia/wikipedia", "20231101.ko", split="train", streaming=True)
    texts = []
    for i, row in enumerate(tqdm(wiki, total=n_docs, desc="위키 수집")):
        if i >= n_docs:
            break
        for line in row["text"].split("\n"):
            line = line.strip()
            if len(line) > 20:   # 너무 짧은 줄 제외
                texts.append(line)
    return texts


wiki_texts = load_wiki_texts(WIKI_DOCS)
print(f"코퍼스 문장 수: {len(wiki_texts):,}")
print("예시:", wiki_texts[0][:60], "...")

## 2. 토크나이저 학습 & 저장

⚠️ 순수 파이썬 BPE 라 vocab 이 크면 느리다. 끝까지 안 돌면 `config.VOCAB_SIZE` 를
낮추거나(예: 8000) `WIKI_DOCS` 를 줄여서 다시 시도. **한 번만 돌리면 되는 작업**이라
완료되면 `tokenizer.json` 을 Drive 에 저장해 두고 다시는 학습하지 않는다.

In [ ]:
TOKENIZER_PATH = "tokenizer.json"

tokenizer = BPETokenizer()
tokenizer.train(wiki_texts, vocab_size=config.VOCAB_SIZE)
tokenizer.save(TOKENIZER_PATH)

# train_model.ipynb 가 읽어갈 수 있도록 Drive 에 고정
shutil.copy(TOKENIZER_PATH, f"{DRIVE_DIR}/tokenizer.json")
print(f"토크나이저 저장 완료. vocab size = {len(tokenizer.vocab)}")
print(f"→ {DRIVE_DIR}/tokenizer.json")

## 3. 검증

In [ ]:
# 라운드트립(원문 복원) + subword 분해가 말이 되는지 확인
text = "한국의 수도는 서울입니다"
ids = tokenizer.encode(text)
print("원문   :", text)
print("인코딩 :", ids)
print("디코딩 :", tokenizer.decode(ids))

for w in ["토크나이저", "위키피디아", "안녕하세요"]:
    print(w, "->", tokenizer._tokenize_word(w))